In [ ]:
import os
import sys
sys.path.append("/workspaces/dev/app")
os.chdir("/workspaces/dev")

In [ ]:
import asyncio
import websockets
import msgpack
import numpy as np
import base64
import requests
import librosa
from IPython.display import Audio
import json

In [ ]:
from util.util import compress_to_opus, np_to_wav

In [ ]:
SAMPLE_RATE = 16000
# BASE_SOCKET_URL = "wss://api.sangjeong.com:8080"
BASE_SOCKET_URL = "ws://localhost:8000"
# BASE_URL = "https://api.sangjeong.com:8080"
BASE_URL = "http://localhost:8000"

# 오디오 로드 및 세그멘팅

In [ ]:
original_audio, sr = librosa.load(".data/boda.wav", sr=SAMPLE_RATE)
# original_audio, sr = librosa.load(".data/news_with_english.wav", sr=SAMPLE_RATE)

In [ ]:
total_samples = len(original_audio)

segments = []
pos = 0
while pos < total_samples:
    rand_len = int(np.random.normal(loc=4000, scale=400))
    rand_len = np.clip(rand_len, 3500, 4500)
    end = min(pos + rand_len, total_samples)

    chunk = original_audio[pos:end]
    segments.append(chunk)
    pos = end

# 오디오 출력해보기

In [ ]:
idx = 0

In [ ]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

# 유틸

In [ ]:
def dump_using_json(data:dict):
    text = json.dumps(data).encode("utf-8")
    length = len(text).to_bytes(4, byteorder="big")
    return length + text

def dump_using_msgpack(data:dict):
    text = msgpack.packb(data)
    length = len(text).to_bytes(4, byteorder="big")
    return length + text

In [ ]:
def dump_audio(audio):
    if audio is None:
        return b''
    audio = np_to_wav(audio, SAMPLE_RATE)
    audio, _ = compress_to_opus(audio)
    return audio

# 임베딩 데이터 만들기

In [ ]:
# 예시 음성 데이터의 일부
refer_dict = {
    "0": [(148061, 339589), (338760, 436145), (433746, 517335)],
    "1": [(1013274, 1068548), (1071733, 1209101), (1209858, 1344749)],
    "2": [(2563542, 2662059), (4373060, 4438192), (4570430, 4625790)],
    "3": [(880324, 938541), (7042377, 7096457), (7106339, 7156845)],
    "4": [(944284, 1009464), (1802918, 1921616), (1921626, 2053013)],
}

In [ ]:
embedding_refer_dict = {"0": [], "1": [], "2": [], "3": [], "4": []}
for idx, ts in refer_dict.items():
    for s in ts:
        audio = original_audio[s[0] : s[1]]
        bt = dump_audio(audio)
        res = requests.post(
            BASE_URL + "/diarization/embed_stream",
            params={"sample_rate": SAMPLE_RATE},
            data=bt,
            headers={"Content-Type": "application/octet-stream"},
        )
        embedding_refer_dict[idx].append(res.json()["embedding"])

In [ ]:
# 임베딩 데이터 확인
embedding = embedding_refer_dict["0"][0]
print(embedding)
print(len(embedding))
bytes_ = base64.b64decode(embedding)
print(len(bytes_))
embedding = np.frombuffer(bytes_, dtype=np.float32)
print(embedding.shape)

# Socket 통신 해보기

In [ ]:
def get_diarization_refer_data(
    group_id:str, refer:dict[str, list[str]], dump:callable =dump_using_msgpack
):
    command = {
        "flag": "diarization_refer", # flag
        "group_id": group_id,
    }
    data =dump(command)
    refer_metadata = {}
    refer_bytes = b''
    for key, value in refer.items():
        refer_metadata[key] = len(value)
        for v in value:
            refer_bytes += base64.b64decode(v)
    data += dump(refer_metadata) + refer_bytes
    return data

def get_metadata_data(
    group_id:str,
    agenda:list[str] | None = None, num_people:int | None = None, meeting_topic:str|None = None,
    dump:callable = dump_using_msgpack
):
    command = {
        "flag": "metadata", # flag
        "group_id": group_id,
    }

    if agenda is not None:
        command["agenda"] = agenda
    if num_people is not None:
        command["num_people"] = num_people
    if meeting_topic is not None:
        command["meeting_topic"] = meeting_topic
    return dump(command)

def get_diarization_data(
    user_id:str, group_id:str, audio:np.ndarray, sc_offset:int | None = None,
    dump:callable = dump_using_msgpack
):
    command = {
        "flag": "diarization", # flag
        "user_id": user_id,
        "group_id": group_id,
    }
    if sc_offset is not None:
        command["sc_offset"] = sc_offset
    data = dump(command)
    audio_bytes = dump_audio(audio)
    data += audio_bytes
    return data

def get_context_data(
    group_id:str,
    dump:callable = dump_using_msgpack
):
    command = {
        "flag": "context",
        "group_id": group_id,
    }
    return dump(command)

def get_context_done_data(
    group_id:str,
    dump:callable = dump_using_msgpack
):
    command = {
        "flag": "context_done",
        "group_id": group_id,
    }
    return dump(command)

In [ ]:
GROUP_ID = "2"

In [ ]:
# import logging
# logging.basicConfig(level=logging.DEBUG)
# logger = logging.getLogger("websockets.protocol")
# logger.setLevel(logging.DEBUG)
async with websockets.connect(
    BASE_SOCKET_URL + "/socket/ws/msgpack", ping_interval=30, ping_timeout=None # 테스트 용
    # JSON 형식으로 원하면
    # BASE_SOCKET_URL + "/socket/ws/json"
) as websocket:
    output = []
    segment_data = [
        (0, 0),
        (1500, sum(len(segment) for segment in segments[:1000])),
    ]

    async def send():
        await websocket.send(
            get_diarization_refer_data(GROUP_ID, embedding_refer_dict)
        )

        await websocket.send(
            get_metadata_data(
                GROUP_ID,
                agenda=["자기소개", "각자 조사 자료 설명", "마무리"],
                num_people=5,
                meeting_topic="신기한 과학 소개 및 설명",
            )
        )

        for idx, segment in enumerate(segments):
            await asyncio.sleep(0)
            for user, (start_idx, sc_offset) in enumerate(segment_data):
                if idx >= start_idx:
                    await websocket.send(
                        get_diarization_data(str(user), GROUP_ID, segment, sc_offset)
                    )

            if idx % 100 == 0:
                print(f"<-------- send {idx} --------")

            if idx != 0 and idx % 1000 == 0:
                await websocket.send(get_context_data(GROUP_ID))

        await websocket.send(get_context_done_data(GROUP_ID))

        print("<-------- send done --------")

    async def receive():
        while True:
            await asyncio.sleep(0)
            result = await websocket.recv()
            response = msgpack.unpackb(result, raw=False)
            # print(response)
            if "completed" in response and "candidate" in response:
                completed = "\n\t".join(
                    f'[{sentence["order"]}][{sentence["words"][0]["start"]} ~ {sentence["words"][-1]["end"]}] {sentence["text"]}'
                    for sentence in response["completed"]
                )
                candidate = "\n\t".join(
                    f'[{sentence["order"]}][{sentence["words"][0]["start"]} ~ {sentence["words"][-1]["end"]}] {sentence["text"]}'
                    for sentence in response["candidate"]
                )
                print("completed: ")
                print(f"\t{completed}")
                print("candidate: ")
                print(f"\t{candidate}")
                print("-" * 30)
            if "context" in response:
                print(response)

            if response and "completed" in response and response["completed"]:
                output.append(response["completed"])

    await asyncio.gather(send(), receive())